# Journal de bord

## 📅 Journal de bord – Jour 1 (18/09/2026)

### 🔧 Étapes / Actions réalisées
- Collections des data
- Organisation du squelette
- Préparation du git

### 📝 Observations / Difficultés
- /

### ✅ Prochaines étapes
- Exploration des données

## 📅 Journal de bord – Jour 2 (27/09/2026)


### 🔧 Étapes / Actions réalisées
- Relecture intégrale du sujet - entretien client, cartographie sources, risques RGPD croisés, datasheet Gebru, grille de décision, traçage d'expériences
- Ajout de 5 compléments ciblés au §1 du notebook : cartographie de la source unique, mécanisme de ré-identification par croisement (`code_insee_commune`), condition de bascule TF-IDF → embeddings, annonce du futur format de traçage


### 📝 Observations / Difficultés
- Notebook resté l'unique livrable (pas de fichier `identification_sources.md` séparé)


### ✅ Prochaines étapes
- Étape 2 : Exploration


## 📅 Journal de bord – Jour 3 (28/09/2026)


### 🔧 Étapes / Actions réalisées
- Mise en place de l'environnement d'exécution
- Audit qualité complet du CSV : manquants confirmés, 74 lignes incohérentes confirmées, code ROME `W1401` identifié comme non conforme à la nomenclature officielle (lettre absente de A-N)
- Vérification PII sur `synthese_entretien` : aucune trouvée (9 phrases-gabarits génériques)
- Calcul du DI sur `nationalite_hors_ue`
- Création de `src/trajectoire_emploi/fairness.py` 
- Brouillon de datasheet Gebru et dictionnaire de variables ajoutés au notebook
- Notebook vérifié exécutable de bout en bout (`nbconvert --execute`, 0 erreur)


### 📝 Observations / Difficultés
- Aucun package Python n'était installé sur la machine au départ 
- `.gitignore` à compléter pour `.venv/`


### ✅ Prochaines étapes
- Étape 3 : Qualité et biais (traitement des 74 lignes incohérentes, code W1401, bilan éthique n°2)


## 📅 Journal de bord – Jour 4 (29/09/2026)


### 🔧 Étapes / Actions réalisées
- Étape 3 réalisée : crosstab cible × nationalité sur 3 classes 
- Rédaction bilan éthique n°2
- Ajout de 4 visualisations EDA — distribution de la cible, distribution de l'âge, boxplot de l'ancienneté (données brutes), heatmap cible × nationalité  

- Étape 4 (Préparation) réalisée : `src/trajectoire_emploi/features.py` (extraction du département depuis le code INSEE, besoin hors ontologie) et `src/trajectoire_emploi/pipeline.py` (4 `ColumnTransformer`, un par scénario) créés, 
- Création de 11 nouveaux tests (`test_features.py`, `test_pipeline.py`)
- Application du split stratifié 2000/500 , pas de split temporel (pas de date dans le jeu de données)

### 📝 Observations / Difficultés
- Bug de chemin `DATA_PATH`/`sys.path` (`../../` vs `../`) non résolu
- Le split n'est pas encore scellé formellement (pas de `outputs/split_scelle.json`)


### ✅ Prochaines étapes
- Étape 5 : Benchmark multi-modèles


## 📅 Journal de bord – Jour 5 (30/09/2026)


### 🔧 Étapes / Actions réalisées
- Étape 5 (Benchmark) réalisée : `src/trajectoire_emploi/evaluation.py` (F1 macro, recall classe 2, kappa pondéré, taux d'erreur 2→0, coût moyen) et `src/trajectoire_emploi/benchmark.py` (boucle scénarios × modèles, mêmes folds)
- Création de 12 nouveaux tests
- Benchmark complet : 4 scénarios × 6 modèles (baseline, régression logistique, RandomForest ×2, LightGBM, XGBoost), `RepeatedStratifiedKFold` (5 folds × 3 répétitions) — meilleur compromis provisoire : LightGBM/XGBoost sur S1 (F1 macro ≈ 0,695)
- Découverte marquante : sur S3 (texte seul), tous les modèles convergent vers un score strictement identique — vérifié : seulement 10 vecteurs TF-IDF distincts (9 phrases-gabarits + 1 pour les manquants)
- Test de robustesse : extrapolation silencieuse détectée sur un âge hors plage d'entraînement (150 ans) ; garde-fou proposé (bornes d'entrée + abstention) pour la future API
- Explication de l'intérêt de la matrice de coûts ajoutée au README

- Étape 6 (Arbitrage) réalisée : `src/trajectoire_emploi/decision.py` (décision à coût minimal) et `src/trajectoire_emploi/calibration.py` (ECE, reliability diagram)
- Création de 6 nouveaux tests
- Vérification de la calibration (ECE ≈ 0,04-0,05) pour les 3 candidats retenus (S1-LightGBM, S1-XGBoost, S2-LightGBM)
- Décision à coût minimal (vs argmax) : divise par plus de deux le taux d'erreur critique 2→0 pour un coût modeste en F1 macro
- Audit d'équité refait sur les **prédictions** (pas l'étiquette) — le modèle **amplifie** le biais sur S1, mais **repasse au-dessus du seuil d'alerte 4/5** sur S2 (DI = 0,838) en retirant seulement `nationalite_hors_ue`
- Choix final tranché : **S2-LightGBM avec décision à coût minimal** — compromis performance/équité argumenté et chiffré

- Étape 7 (Communication) réalisée : `decision.py` enrichi de `cout_attendu_minimal()` et `decision_avec_abstention()` (5 tests) ; analyse d'erreurs (bloc principal = confusion 0↔1, pas 2→0) ; seuil d'abstention retenu à 0,7 (~19 % de revue humaine, aligné sur < 20 %) 
- Rédaction de la note de recommandation client
- **Verdict final sur le test scellé** : F1 macro = 0,722, taux d'erreur critique 2→0 = 0,027, DI sur les prédictions = 0,808 (toujours au-dessus du seuil d'alerte 4/5)
- Traçage du scellement dans `outputs/split_scelle.json` (modèle, seuil, métriques, date d'exécution)

- Étape 8 (FastAPI) réalisée
- Création de `persistence.py` (`construire_metadata`, `persister_modele`, `charger_modele`) — packaging du pipeline complet (préprocesseur + classifieur), 5 clés obligatoires dans le `.json` adjacent (4 tests)
- **Réentraînement final** du pipeline S2-LightGBM sur **train+test combinés** (2500 lignes)
- Packaging : `models/trajectoire_emploi_v1.joblib` + `.json`
- API FastAPI créée (`app/main.py`, `app/schemas.py`, `app/middleware.py`) : `/health`, `/info`, `/predict` avec décision à coût minimal + abstention (seuil 0,7) ; logs structurés avec `request_id`, sans PII
- 10 tests API + contract test du modèle ajoutés — 55 tests passent au total

- Remise en cause de l'argument d'équité : le DI utilisé partout mesurait « prédit classe 0 » comme issue favorable, alors que §3.6 définit le préjudice comme la privation d'accompagnement (classe 2 non détectée) — convention inversée par rapport au préjudice réel
- Refait entièrement : (1a) comparaison taux réel vs prédit de classe 0 par groupe → révèle que S2 prédit « retour rapide » pour le groupe hors-UE **3× plus souvent que la réalité** (31,8 % prédit vs 13,9 % réel) — le DI remonte par perte de pouvoir discriminant, pas par correction ; (1b) recall classe 2 et taux 2→0 **par groupe**, sous la vraie règle : **renversement complet** — S1 protège mieux le groupe hors-UE (recall 0,804, taux 2→0 0,011) que le groupe majoritaire ; S2 inverse ce résultat (recall 0,620, taux 2→0 **0,076, 7× pire qu'avec S1**)
- DI recalculé sous la bonne règle : 0,722 (pas 0,838) pour la convention classe 0, 0,825 pour la convention classe 2 (accompagnement) — les deux nuances absentes du chiffre original
- Création de `experiments.md` (racine, 5 runs documentés de la baseline au modèle de production)

### 📝 Observations / Difficultés
- Bug de chemin `DATA_PATH`/`sys.path` (`../../` vs `../`) non résolu

### ✅ Prochaines étapes
- Étape 9 (Docker) → Étape 10 (CI/CD)
